# Simplified version of model

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, concatenate, ReLU, Input, Conv2DTranspose 
from tensorflow.keras.layers import Add, Flatten, Dense, Layer, Lambda, Concatenate, Reshape, Multiply

# Buat model segmentasi dengan feature extractor
input_shape = (128, 128, 3)
batch_sz = 2

In [2]:
import os
import tensorflow as tf
import pandas as pd

def load_data(image_dir, mask_dir, gt_dir, image_size=(256, 256)):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith('.jpg')])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) if fname.endswith('.png')])
    gt_paths = sorted([os.path.join(gt_dir, fname) for fname in os.listdir(gt_dir) if fname.endswith('.csv')])

    def load_image_mask_gt(image_path, mask_path, gt_path):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0

        mask = tf.io.read_file(mask_path)
        mask = tf.image.decode_png(mask, channels=1)
        mask = tf.image.resize(mask, image_size)
        mask = tf.cast(mask, tf.float32) / 255.0

        # gt = pd.read_csv(gt_path).values
        # gt = tf.convert_to_tensor(gt, dtype=tf.float32)
        gt_content = tf.io.read_file(gt_path)
        gt_lines = tf.strings.split(gt_content, '\n')
        gt_values = tf.strings.split(gt_lines[1], ',')
        gt = tf.strings.to_number(gt_values, tf.float32)

        return image, mask, gt

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths, gt_paths))
    dataset = dataset.map(lambda img, msk, gt: tf.py_function(load_image_mask_gt, [img, msk, gt], [tf.float32, tf.float32, tf.float32]), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

def create_dataloader(data_dir, image_size=(256, 256), batch_size=8):
    train_image_dir = os.path.join(data_dir, 'train', 'images')
    train_mask_dir = os.path.join(data_dir, 'train', 'masks')
    train_gt_dir = os.path.join(data_dir, 'train', 'groundtruth')
    
    val_image_dir = os.path.join(data_dir, 'validation', 'images')
    val_mask_dir = os.path.join(data_dir, 'validation', 'masks')
    val_gt_dir = os.path.join(data_dir, 'validation', 'groundtruth')
    
    test_image_dir = os.path.join(data_dir, 'test', 'images')
    test_mask_dir = os.path.join(data_dir, 'test', 'masks')
    test_gt_dir = os.path.join(data_dir, 'test', 'groundtruth')

    train_dataset = load_data(train_image_dir, train_mask_dir, train_gt_dir, image_size)
    val_dataset = load_data(val_image_dir, val_mask_dir, val_gt_dir, image_size)
    test_dataset = load_data(test_image_dir, test_mask_dir, test_gt_dir, image_size)

    train_dataset = train_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    val_dataset = val_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    test_dataset = test_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return train_dataset, val_dataset, test_dataset

# Example usage
data_dir = 'data/dataset_b300'
train_dataset, val_dataset, test_dataset = create_dataloader(data_dir, image_size=(640, 640), batch_size=batch_sz)
print(train_dataset.take(1))
for images, masks, gts in train_dataset.take(1):
    print("Images shape:", images.shape)
    print("Masks shape:", masks.shape)
    print("Ground truths shape:", gts.shape)
    print("Sample ground truth values:", gts.numpy()[0])  # Print the first sample for verification


<TakeDataset element_spec=(TensorSpec(shape=<unknown>, dtype=tf.float32, name=None), TensorSpec(shape=<unknown>, dtype=tf.float32, name=None), TensorSpec(shape=<unknown>, dtype=tf.float32, name=None))>
Images shape: (2, 640, 640, 3)
Masks shape: (2, 640, 640, 1)
Ground truths shape: (2, 7)
Sample ground truth values: [ 0.00383671 -0.03949213  0.430923   -0.07138272  0.03482315  0.08684116
  0.9930511 ]


## Old way

In [24]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import Sequence

class ImageGenerator(Sequence):
    def __init__(self, image_dir, batch_size=32, img_size=(256, 256), shuffle=True):
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.image_files = sorted(os.listdir(image_dir))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.floor(len(self.image_files) / self.batch_size))
    
    def __getitem__(self, index):
        # Generate indexes of the batch
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        
        # Find list of image and groundtruth files
        image_files_batch = [self.image_files[k] for k in indexes]
        
        # Generate data
        X = self.__data_generation(image_files_batch)
        return X, np.array(1)
        
    def on_epoch_end(self):
        self.indexes = np.arange(len(self.image_files))
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __data_generation(self, image_files_batch):
        images = []
        
        for img_file in (image_files_batch):
            img_path = os.path.join(self.image_dir, img_file)
            
            try:
                img = load_img(img_path, target_size=self.img_size)
            except Exception as e:
                logging.error(f"Error loading image or groundtruth: {img_path}. Error: {e}")
                continue
            
            img_array = img_to_array(img) / 255.0
            
            images.append(img_array)
        
        return np.array(images)
    
    def get_images_only(self):
        images = []
        for img_file in self.image_files:
            img_path = os.path.join(self.image_dir, img_file)
            try:
                img = load_img(img_path, target_size=self.img_size)
            except Exception as e:
                logging.error(f"Error loading image: {img_path}. Error: {e}")
                continue
            img_array = img_to_array(img) / 255.0
            images.append(img_array)
        return np.array(images)

    def get_images_only_batch(self):
        total_images = self.get_images_only()
        num_batches = len(total_images) // self.batch_size
        for i in range(num_batches):
            yield total_images[i * self.batch_size:(i + 1) * self.batch_size]

class GroundTruthGenerator(Sequence):
    def __init__(self, groundtruth_dir, batch_size=32, img_size=(256, 256), shuffle=True):
        self.groundtruth_dir = groundtruth_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.groundtruth_files = sorted(os.listdir(groundtruth_dir))
        self.indexes = np.arange(len(self.groundtruth_files))
    
    def __len__(self):
        return int(np.floor(len(self.groundtruth_files) / self.batch_size))       
    
    def __getitem__(self, index):
        # Generate indexes of the batch
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        
        # Find list of groundtruth files
        groundtruth_files_batch = [self.groundtruth_files[k] for k in indexes]
        
        # Generate data
        y = self.__data_generation(groundtruth_files_batch)
        return y, np.array(1)
    
    def __data_generation(self, groundtruth_files_batch):
        groundtruths = []
        
        for gt_file in groundtruth_files_batch:
            gt_path = os.path.join(self.groundtruth_dir, gt_file)
            
            try:
                gt = pd.read_csv(gt_path).values  # Memuat groundtruth dengan pandas
            except Exception as e:
                logging.error(f"Error loading image or groundtruth: {gt_path}. Error: {e}")
                continue
            
            groundtruths.append(gt)
        return np.array(groundtruths, dtype = np.float32) 
            
# Inisialisasi generator untuk train dan validation
image_train_gen = ImageGenerator('data/dataset_b300/train/images', batch_size=batch_sz, 
                                 img_size = (128, 128), shuffle=False)
gtruth_train_gen = GroundTruthGenerator('data/dataset_b300/train/groundtruth', batch_size=batch_sz, 
                                        img_size = (128, 128), shuffle=False)
image_valid_gen = ImageGenerator('data/dataset_b300/validation/images', batch_size=batch_sz,
                                img_size = (128, 128), shuffle=False)
gtruth_valid_gen = GroundTruthGenerator('data/dataset_b300/validation/groundtruth', batch_size=batch_sz, 
                                        img_size = (128, 128), shuffle=False)

print(len(image_train_gen))
print(type(image_train_gen))
print(len(gtruth_train_gen))
print(type(gtruth_train_gen))
print(len(image_valid_gen))
print(type(image_valid_gen))
print(len(gtruth_valid_gen))
print(len(gtruth_valid_gen[0][0][0][0]))

885
<class '__main__.ImageGenerator'>
885
<class '__main__.GroundTruthGenerator'>
252
<class '__main__.ImageGenerator'>
252
7


In [40]:
class ApplyThreshold(Layer):
    def call(self, x):
        return tf.where(x>0.5, 1., 0.)

In [44]:
from tensorflow.keras.applications import VGG16

# Define the feature extraction part using VGG16
def noob_version(input_shape):
    # Input untuk RGB image
    rgb_input = Input(shape=input_shape)
    tf.print(rgb_input.shape)
    # Load the VGG16 model without the top layers
    vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze the VGG16 layers
    for layer in vgg16.layers:
        layer.trainable = False
    
    # Extract specific layers to match the diagram
    block4_conv3 = vgg16.get_layer('block4_conv3').output
    block5_conv3 = vgg16.get_layer('block5_conv3').output

    
    '''
        Decoder for segmentation layer
    '''
    # block5_conv3 upsampling
    SBk = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(block5_conv3)  # 1/16
    # tf.print(SBk.shape)
    SBk = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(SBk) # Deconv layer
    SBk = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(SBk)  # 1/8
    # tf.print(SBk.shape)
    # block4_conv3 upsampling
    SBb = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(block4_conv3) # 1/8
    # tf.print(SBb.shape)
    xS  = Add()([SBk, SBb])
    
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer  # 1/4
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/2
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/1
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    
    # Final segmentation layer
    segmentation_output = Conv2D(1, kernel_size=(1, 1), activation='sigmoid', padding='same')(xS)
    # segmentation = tf.where(segmentation_output>0.5, 1., 0.)
    segmentation = ApplyThreshold()(segmentation_output)
    
    '''
        Decoder for depth regression
    '''
    # block5_conv3 upsampling
    DBk = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(block5_conv3)  # 1/16
    DBk = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(DBk) # Deconv layer
    DBk = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(DBk)  # 1/8
    # block4_conv3 upsampling
    DBb = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(block4_conv3) # 1/8
    xD  = Add()([DBk, DBb])
    
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer  # 1/4
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/2
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/1
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    
    # Final depth layer
    depth_regression = Conv2D(4, kernel_size=(1, 1), activation='sigmoid', padding='same')(xS)

    focused_area = depth_regression * segmentation
    x = Flatten()(focused_area)
    x = Dense(128, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    translations = Dense(3, activation='linear')(x) 
    orientations = Dense(4, activation='linear')(x)
    outputs = Concatenate()([translations, orientations])
    tf.print("Concatenate out:", outputs.shape)
    outputs = Reshape((1, 7))(outputs)
    tf.print("outputs out:", outputs.shape)
    
    model = Model(inputs= rgb_input, outputs = outputs)
    
    return model
    
model = noob_version(input_shape)
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mse'])

(None, 128, 128, 3)
Concatenate out: (None, 7)
outputs out: (None, 1, 7)


In [43]:
model.summary()

Model: "functional_33"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_38      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 128, 128,  │      1,792 │ input_layer_38[0… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 128, 128,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 64, 64,    │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 64, 64,    │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 64, 64,    │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 32, 32,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 32, 32,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 32, 32,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 32, 32,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 16, 16,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 16, 16,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 16, 16,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 16, 16,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 8, 8, 512) │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 8, 8, 512) │  2,359,808 │ block4_pool[0][0] │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv2        │ (None, 8, 8, 512) │  2,359,808 │ block5_conv1[0][

 Total params: 24,080,396 (91.86 MB)

 Trainable params: 24,080,396 (91.86 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.fit(
    train_dataset,
    epochs=1,
    batch_size=batch_sz
)

KeyError: 'Exception encountered when calling Functional.call().\n\n\x1b[1m139873133247744\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=tf.Tensor(shape=(None, 128, 128, 3), dtype=float32)\n  • training=True\n  • mask=None'

In [36]:
import os
import tensorflow as tf
import pandas as pd

def load_data(image_dir, groundtruth_dir, image_size=(256, 256)):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith('.jpg')])
    groundtruth_paths = sorted([os.path.join(groundtruth_dir, fname) for fname in os.listdir(groundtruth_dir) if fname.endswith('.csv')])

    def load_image_and_groundtruth(image_path, groundtruth_path):
        image_path = image_path.numpy().decode('utf-8')
        groundtruth_path = groundtruth_path.numpy().decode('utf-8')

        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0

        groundtruth_df = pd.read_csv(groundtruth_path)
        groundtruth_values = groundtruth_df.iloc[0].values  # Only take the second row
        groundtruth = tf.convert_to_tensor(groundtruth_values, dtype=tf.float32)

        return image, groundtruth

    def tf_load_image_and_groundtruth(image_path, groundtruth_path):
        image, groundtruth = tf.py_function(
            func=load_image_and_groundtruth,
            inp=[image_path, groundtruth_path],
            Tout=(tf.float32, tf.float32)
        )
        image.set_shape([*image_size, 3])
        groundtruth.set_shape([7])
        return image, groundtruth

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, groundtruth_paths))
    dataset = dataset.map(tf_load_image_and_groundtruth, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

def create_dataloader(data_dir, image_size=(256, 256), batch_size=8):
    train_image_dir = os.path.join(data_dir, 'train', 'images')
    train_groundtruth_dir = os.path.join(data_dir, 'train', 'groundtruth')
    val_image_dir = os.path.join(data_dir, 'validation', 'images')
    val_groundtruth_dir = os.path.join(data_dir, 'validation', 'groundtruth')

    train_dataset = load_data(train_image_dir, train_groundtruth_dir, image_size)
    val_dataset = load_data(val_image_dir, val_groundtruth_dir, image_size)

    train_dataset = train_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    val_dataset = val_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return train_dataset, val_dataset

# Example usage
data_dir = 'data/dataset_b300'
train_dataset_f, val_dataset = create_dataloader(data_dir, image_size=(128, 128), batch_size=batch_sz)

for images, groundtruths in train_dataset_f.take(1):
    print(images.shape)
    print(groundtruths.shape)
    # print(groundtruths)
print(train_dataset_f.element_spec)
train_dataset_f = train_dataset_f.map(lambda x, y: (tf.cast(x, tf.float32), y))

(4, 128, 128, 3)
(4, 7)
(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 7), dtype=tf.float32, name=None))


2024-07-06 16:09:29.619939: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [37]:
noob_model.summary()

Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_23      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 128, 128,  │      1,792 │ input_layer_23[0… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 128, 128,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 64, 64,    │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 64, 64,    │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 64, 64,    │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 32, 32,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 32, 32,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 32, 32,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 32, 32,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 16, 16,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 16, 16,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 16, 16,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 16, 16,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 8, 8, 512) │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 8, 8, 512) │  2,359,808 │ block4_pool[0][0] │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv2        │ (None, 8, 8, 512) │  2,359,808 │ block5_conv1[0][

 Total params: 24,080,396 (91.86 MB)

 Trainable params: 9,365,708 (35.73 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [38]:
# Generate random data
batch_sz = 1
x_train_rgb = tf.random.uniform((batch_sz, 128, 128, 3), minval=0, maxval=1)
print(x_train_rgb.shape)
y_train_translations = tf.random.uniform((batch_sz, 3), minval=0, maxval=1)
y_train_orientations = tf.random.uniform((batch_sz, 4), minval=0, maxval=1)
y_train = tf.concat([y_train_translations, y_train_orientations], axis=1)
y_train = tf.reshape(y_train, (1, 7))

print(f"x_train_rgb shape: {x_train_rgb.shape}")

print(f"y_train shape: {y_train.shape}")

# Create a dataset from the tensors
train_dataset = tf.data.Dataset.from_tensor_slices((
    x_train_rgb, 
    y_train
)).batch(batch_sz)

# Train the model
input_shape = (128, 128, 3)
noob_model = noob_version(input_shape)
noob_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mse'])

noob_model.fit(
    train_dataset,
    epochs=3,
    batch_size=batch_sz
)

(1, 128, 128, 3)
x_train_rgb shape: (1, 128, 128, 3)
y_train shape: (1, 7)
(None, 128, 128, 3)
Concatenate out: (None, 7)
outputs out: (None, 1, 7)
Epoch 1/3


KeyError: 'Exception encountered when calling Functional.call().\n\n\x1b[1m139873463297040\x1b[0m\n\nArguments received by Functional.call():\n  • inputs=tf.Tensor(shape=(None, 128, 128, 3), dtype=float32)\n  • training=True\n  • mask=None'

In [28]:
tf.__version__

'2.16.1'

In [29]:
%pip install tensorflow==2.10

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.0/578.0 MB 3.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 43.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.7/438.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.3/781.3 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 17.4 MB/s eta 0:00:00
  Attempting unin

In [45]:
def simpleCNN(input_shape):
    rgb_input = Input(shape=input_shape)
    x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(rgb_input)
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = Conv2D(256, kernel_size=())

## New way

### dataloader

In [15]:
import os
import tensorflow as tf
import pandas as pd

def load_data(image_dir, groundtruth_dir, image_size=(256, 256)):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith('.jpg')])
    groundtruth_paths = sorted([os.path.join(groundtruth_dir, fname) for fname in os.listdir(groundtruth_dir) if fname.endswith('.csv')])

    def load_image_and_groundtruth(image_path, groundtruth_path):
        image_path = image_path.numpy().decode('utf-8')
        groundtruth_path = groundtruth_path.numpy().decode('utf-8')

        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0

        groundtruth_df = pd.read_csv(groundtruth_path)
        groundtruth_values = groundtruth_df.iloc[0].values  # Only take the second row
        groundtruth = tf.convert_to_tensor(groundtruth_values, dtype=tf.float32)

        return image, groundtruth

    def tf_load_image_and_groundtruth(image_path, groundtruth_path):
        image, groundtruth = tf.py_function(
            func=load_image_and_groundtruth,
            inp=[image_path, groundtruth_path],
            Tout=(tf.float32, tf.float32)
        )
        image.set_shape([*image_size, 3])
        groundtruth.set_shape([7])
        return image, groundtruth

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, groundtruth_paths))
    dataset = dataset.map(tf_load_image_and_groundtruth, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

def create_dataloader(data_dir, image_size=(256, 256), batch_size=8):
    train_image_dir = os.path.join(data_dir, 'train', 'images')
    train_groundtruth_dir = os.path.join(data_dir, 'train', 'groundtruth')
    val_image_dir = os.path.join(data_dir, 'validation', 'images')
    val_groundtruth_dir = os.path.join(data_dir, 'validation', 'groundtruth')

    train_dataset = load_data(train_image_dir, train_groundtruth_dir, image_size)
    val_dataset = load_data(val_image_dir, val_groundtruth_dir, image_size)

    train_dataset = train_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    val_dataset = val_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return train_dataset, val_dataset

# Example usage
data_dir = 'data/dataset_b300'
train_dataset_f, val_dataset = create_dataloader(data_dir, image_size=(128, 128), batch_size=batch_sz)

for images, groundtruths in train_dataset_f.take(1):
    print(images.shape)
    print(groundtruths.shape)
    print(groundtruths)
print(train_dataset_f.element_spec)
train_dataset_f = train_dataset_f.map(lambda x, y: (tf.cast(x, tf.float32), y))

(2, 128, 128, 3)
(2, 7)
tf.Tensor(
[[ 0.00383671 -0.03949213  0.430923   -0.07138272  0.03482315  0.08684116
   0.9930511 ]
 [ 0.00712653 -0.03169478  0.42852286  0.07186151  0.08804508  0.1014372
   0.9883291 ]], shape=(2, 7), dtype=float32)
(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 7), dtype=tf.float32, name=None))


### Thresholding Layer

In [17]:
class ApplyThreshold(Layer):
    def call(self, x):
        return tf.where(x>0.5, 1., 0.)

### model original

In [3]:
from tensorflow.keras.applications import VGG16

# Define the feature extraction part using VGG16
def noob_version(input_shape):
    # Input untuk RGB image
    rgb_input = Input(shape=input_shape)

    '''
        Encoder following the VGG-16 model
    '''
    # block 1
    x = Conv2D(64, kernel_size = (3, 3), activation='relu', padding='same')(rgb_input)
    x = Conv2D(64, kernel_size = (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 2
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 3
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 4
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x1 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    # block4_conv3
    x2 = MaxPooling2D(pool_size=2, strides=2)(x1)    
    # block 5
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    # block5_conv3
    
    # Segmentation Bottleneck operations
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    x2s = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(x2s) # Deconv layer
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2s)  # 1/8
    x1s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    xS  = Add()([x1s, x2s])
    # Decoder
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer  # 1/4
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/2
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/1
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    
    # bottleneck depth layers
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    xK2 = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xK2) # Deconv layer
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xK2)  # 1/8
    xB2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    xD  = Add()([xK2, xB2])
    # Decoder
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer  # 1/4
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/2
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/1
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)

    # Final segmentation layer
    xS = Conv2D(1, kernel_size=(1, 1), activation='sigmoid', padding='same')(xS)
    # xS = ApplyThreshold()(xS)
    # Final depth layer
    xD = Conv2D(4, kernel_size=(1, 1), activation='sigmoid', padding='same')(xD)

    xo = Multiply()([xS, xD])

    xo = Flatten()(xo)
    xo = Dense(128, activation='relu')(xo)
    xo = Dense(128, activation='relu')(xo)
    
    xa = Dense(1, activation='linear', name='xa')(xo) 
    ya = Dense(1, activation='linear', name='ya')(xo) 
    za = Dense(1, activation='linear', name='za')(xo) 
    rw = Dense(1, activation='linear', name='rw')(xo) 
    rx = Dense(1, activation='linear', name='rx')(xo) 
    ry = Dense(1, activation='linear', name='ry')(xo)
    rz = Dense(1, activation='linear', name='rz')(xo)
    
    model = Model(inputs= rgb_input, outputs = [xa, ya, za, rw, rx, ry, rz])
    
    return model
    

### model w upsampling

In [24]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, concatenate, ReLU, Input, UpSampling2D 

# Define the feature extraction part using VGG16
# Define the feature extraction part using VGG16
def noob_version(input_shape):
    # Input for RGB image
    rgb_input = Input(shape=input_shape)

    '''
        Encoder following the VGG-16 model
    '''
    # block 1
    x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(rgb_input)
    print(f'block 1, conv2d_1 shape: {x.shape}')
    x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 1, conv2d_2 shape: {x.shape}')
    x = MaxPooling2D(pool_size=2, strides=2)(x)
    print(f'block 1, maxpool shape: {x.shape}')
    # block 2
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 2, conv2d_1 shape: {x.shape}')
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 2, conv2d_2 shape: {x.shape}')
    x = MaxPooling2D(pool_size=2, strides=2)(x)
    print(f'block 2, maxpool shape: {x.shape}')
    # block 3
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 3, conv2d_1 shape: {x.shape}')
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 3, conv2d_2 shape: {x.shape}')
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 3, conv2d_3 shape: {x.shape}')
    x = MaxPooling2D(pool_size=2, strides=2)(x)
    print(f'block 3, maxpool shape: {x.shape}')
    # block 4
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 4, conv2d_1 shape: {x.shape}')
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)
    print(f'block 4, conv2d_2 shape: {x.shape}')
    x1 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    # block4_conv3
    print(f'block 4, conv2d_3 shape: {x1.shape}')
    x2 = MaxPooling2D(pool_size=2, strides=2)(x1)
    print(f'block 4, maxpool shape: {x2.shape}')
    # block 5
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)
    print(f'block 5, conv2d_1 shape: {x2.shape}')
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)
    print(f'block 5, conv2d_2 shape: {x2.shape}')
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)
    print(f'block 5, conv2d_3 shape: {x2.shape}')
    
    # Segmentation Bottleneck operations
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    print(f'bottleneck, conv2d_1 shape: {x2s.shape}')
    x2s = UpSampling2D(size=(2, 2))(x2s) # UpSampling layer
    print(f'bottleneck, upsampling_1 shape: {x2s.shape}')
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2s)  # 1/8
    print(f'bottleneck, conv2d_2 shape: {x2s.shape}')
    x1s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    print(f'bottleneck, conv2d_3 shape: {x1s.shape}')
    xS  = Add()([x1s, x2s])
    print(f'bottleneck, add shape: {xS.shape}')
    # Decoder
    xS  = UpSampling2D(size=(2, 2))(xS) # UpSampling layer  # 1/4
    print(f'decoder, upsampling_1 shape: {xS.shape}')
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    print(f'decoder, conv2d_1 shape: {xS.shape}')
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    print(f'decoder, conv2d_2 shape: {xS.shape}')
    xS  = UpSampling2D(size=(2, 2))(xS) # UpSampling layer # 1/2
    print(f'decoder, upsampling_2 shape: {xS.shape}')
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    print(f'decoder, conv2d_3 shape: {xS.shape}')
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    print(f'decoder, conv2d_4 shape: {xS.shape}')
    xS  = UpSampling2D(size=(2, 2))(xS) # UpSampling layer # 1/1
    print(f'decoder, upsampling_3 shape: {xS.shape}')
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    print(f'decoder, conv2d_5 shape: {xS.shape}')
    
    # bottleneck depth layers
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    print(f'bottleneck depth, conv2d_1 shape: {xK2.shape}')
    xK2 = UpSampling2D(size=(2, 2))(xK2) # UpSampling layer
    print(f'bottleneck depth, upsampling_1 shape: {xK2.shape}')
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xK2)  # 1/8
    print(f'bottleneck depth, conv2d_2 shape: {xK2.shape}')
    xB2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    print(f'bottleneck depth, conv2d_3 shape: {xB2.shape}')
    xD  = Add()([xK2, xB2])
    print(f'bottleneck depth, add shape: {xD.shape}')
    # Decoder
    xD  = UpSampling2D(size=(2, 2))(xD) # UpSampling layer  # 1/4
    print(f'bottleneck depth decoder, upsampling_1 shape: {xD.shape}')
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    print(f'bottleneck depth decoder, conv2d_1 shape: {xD.shape}')
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    print(f'bottleneck depth decoder, conv2d_2 shape: {xD.shape}')
    xD  = UpSampling2D(size=(2, 2))(xD) # UpSampling layer # 1/2
    print(f'bottleneck depth decoder, upsampling_2 shape: {xD.shape}')
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    print(f'bottleneck depth decoder, conv2d_3 shape: {xD.shape}')
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    print(f'bottleneck depth decoder, conv2d_4 shape: {xD.shape}')
    xD  = UpSampling2D(size=(2, 2))(xD) # UpSampling layer # 1/1
    print(f'bottleneck depth decoder, upsampling_3 shape: {xD.shape}')
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    print(f'bottleneck depth decoder, conv2d_5 shape: {xD.shape}')

    # Final segmentation layer
    xS = Conv2D(1, kernel_size=(1, 1), activation='sigmoid', padding='same')(xS)
    print(f'final segmentation layer, conv2d shape: {xS.shape}')
    # xS = ApplyThreshold()(xS)
    # print(f'final segmentation layer, apply threshold shape: {xS.shape}')
    # Final depth layer
    xD = Conv2D(7, kernel_size=(1, 1), activation='sigmoid', padding='same')(xD)
    print(f'final depth layer, conv2d shape: {xD.shape}')

    xo = Multiply()([xS, xD])
    print(f'multiply shape: {xo.shape}')

    xo = Flatten()(xo)
    print(f'flatten shape: {xo.shape}')
    xo = Dense(128, activation='relu')(xo)
    print(f'dense 1 shape: {xo.shape}')
    xo = Dense(128, activation='relu')(xo)
    print(f'dense 2 shape: {xo.shape}')
    
    xa = Dense(7, activation='linear', name='xa')(xo) 
    print(f'xa shape: {xa.shape}')
    # ya = Dense(1, activation='linear', name='ya')(xo) 
    # print(f'ya shape: {ya.shape}')
    # za = Dense(1, activation='linear', name='za')(xo) 
    # print(f'za shape: {za.shape}')
    # rw = Dense(1, activation='linear', name='rw')(xo) 
    # print(f'rw shape: {rw.shape}')
    # rx = Dense(1, activation='linear', name='rx')(xo) 
    # print(f'rx shape: {rx.shape}')
    # ry = Dense(1, activation='linear', name='ry')(xo)
    # print(f'ry shape: {ry.shape}')
    # rz = Dense(1, activation='linear', name='rz')(xo)
    # print(f'rz shape: {rz.shape}')
    
    model = Model(inputs=rgb_input, outputs=xa)
    
    return model


### compile & train

In [25]:
model = noob_version(input_shape)

block 1, conv2d_1 shape: (None, 128, 128, 64)
block 1, conv2d_2 shape: (None, 128, 128, 64)
block 1, maxpool shape: (None, 64, 64, 64)
block 2, conv2d_1 shape: (None, 64, 64, 128)
block 2, conv2d_2 shape: (None, 64, 64, 128)
block 2, maxpool shape: (None, 32, 32, 128)
block 3, conv2d_1 shape: (None, 32, 32, 256)
block 3, conv2d_2 shape: (None, 32, 32, 256)
block 3, conv2d_3 shape: (None, 32, 32, 256)
block 3, maxpool shape: (None, 16, 16, 256)
block 4, conv2d_1 shape: (None, 16, 16, 512)
block 4, conv2d_2 shape: (None, 16, 16, 512)
block 4, conv2d_3 shape: (None, 16, 16, 512)
block 4, maxpool shape: (None, 8, 8, 512)
block 5, conv2d_1 shape: (None, 8, 8, 512)
block 5, conv2d_2 shape: (None, 8, 8, 512)
block 5, conv2d_3 shape: (None, 8, 8, 512)
bottleneck, conv2d_1 shape: (None, 8, 8, 64)
bottleneck, upsampling_1 shape: (None, 16, 16, 64)
bottleneck, conv2d_2 shape: (None, 16, 16, 64)
bottleneck, conv2d_3 shape: (None, 16, 16, 64)
bottleneck, add shape: (None, 16, 16, 64)
decoder, upsam

In [26]:
model.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_5 (InputLayer)           [(None, 128, 128, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_124 (Conv2D)            (None, 128, 128, 64  1792        ['input_5[0][0]']                
                                )                                                                 
                                                                                                  
 conv2d_125 (Conv2D)            (None, 128, 128, 64  36928       ['conv2d_124[0][0]']             
                                )                                                           

In [22]:
model.compile(optimizer='adam', 
              loss={
                  'xa': 'mse', 
                  'ya': 'mse', 
                  'za': 'mse',
                  'rw': 'mse',
                  'rx': 'mse',
                  'ry': 'mse',
                  'rz': 'mse'},
              metrics={
                  'xa': tf.keras.metrics.RootMeanSquaredError(), 
                  'ya': tf.keras.metrics.RootMeanSquaredError(),
                  'za': tf.keras.metrics.RootMeanSquaredError(),
                  'rw': tf.keras.metrics.RootMeanSquaredError(),
                  'rx': tf.keras.metrics.RootMeanSquaredError(),
                  'ry': tf.keras.metrics.RootMeanSquaredError(),
                  'rz': tf.keras.metrics.RootMeanSquaredError()})

In [34]:
# model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mse'])
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError(), metrics=[tf.keras.metrics.RootMeanSquaredError(),
                                                                                  tf.keras.metrics.MeanAbsoluteError()])

In [35]:
model.fit(train_dataset, epochs=3, batch_size=batch_sz)

Epoch 1/3


InvalidArgumentError: Graph execution error:

Detected at node 'gradient_tape/model_4/add_9/add/BroadcastGradientArgs' defined at (most recent call last):
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\runpy.py", line 193, in _run_module_as_main
      "__main__", mod_spec)
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\runpy.py", line 85, in _run_code
      exec(code, run_globals)
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\site-packages\traitlets\config\application.py", line 1043, in launch_instance
      app.start()
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\site-packages\ipykernel\kernelapp.py", line 712, in start
      self.io_loop.start()
    File "c:\Users\diona\anaconda3\envs\py3.7-poseCNN\lib\site-packages\tornado\platform\asyncio.py", line 215, in start
      self.asyncio_loop.run

In [ ]:
from tensorflow.keras.applications import VGG16

# Define the feature extraction part using VGG16
def noob_version(input_shape):
    # Input untuk RGB image
    rgb_input = Input(shape=input_shape)

    '''
        Encoder following the VGG-16 model
    '''
    # block 1
    x = Conv2D(64, kernel_size = (3, 3), activation='relu', padding='same')(rgb_input)
    x = Conv2D(64, kernel_size = (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 2
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 3
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = MaxPooling2D(pool_size=2, strides=2)(x)    
    # block 4
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    
    x1 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x)    # block4_conv3
    x2 = MaxPooling2D(pool_size=2, strides=2)(x1)    
    # block 5
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    
    x2 = Conv2D(512, kernel_size=(3, 3), activation='relu', padding='same')(x2)    # block5_conv3
    
    # Segmentation Bottleneck operations
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    x2s = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(x2s) # Deconv layer
    x2s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x2s)  # 1/8
    x1s = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    xS  = Add()([x1s, x2s])
    # Decoder
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer  # 1/4
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/2
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    xS  = Conv2DTranspose(64, kernel_size=(3, 3), strides=2, padding='same')(xS) # Deconv layer # 1/1
    xS  = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(xS)
    
    # bottleneck depth layers
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x2)  # 1/16
    xK2 = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xK2) # Deconv layer
    xK2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xK2)  # 1/8
    xB2 = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x1) # 1/8
    xD  = Add()([xK2, xB2])
    # Decoder
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer  # 1/4
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/2
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)
    xD  = Conv2DTranspose(128, kernel_size=(3, 3), strides=2, padding='same')(xD) # Deconv layer # 1/1
    xD  = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(xD)

    # Final segmentation layer
    xS = Conv2D(1, kernel_size=(1, 1), activation='sigmoid', padding='same')(xS)
    xS = ApplyThreshold()(xS)
    # Final depth layer
    xD = Conv2D(4, kernel_size=(1, 1), activation='sigmoid', padding='same')(xD)

    xo = Multiply()([xS, xD])

    xo = Flatten()(xo)
    xo = Dense(128, activation='relu')(xo)
    xo = Dense(128, activation='relu')(xo)
    
    xa = Dense(1, activation='linear')(xo) 
    ya = Dense(1, activation='linear')(xo) 
    za = Dense(1, activation='linear')(xo) 
    rw = Dense(1, activation='linear')(xo) 
    rx = Dense(1, activation='linear')(xo) 
    ry = Dense(1, activation='linear')(xo)
    rz = Dense(1, activation='linear')(xo)
    
    model = Model(inputs= rgb_input, outputs = [xa, ya, za, rw, rx, ry, rz])
    
    return model

model = noob_version(input_shape)
model.compile(optimizer='adam', 
              loss={
                  'dense_23': 'mse', 
                  'dense_24': 'mse', 
                  'dense_25': 'mse',
                  'dense_26': 'mse',
                  'dense_27': 'mse',
                  'dense_28': 'mse',
                  'dense_29': 'mse'},
              metrics={
                  'dense_23': tf.keras.metrics.RootMeanSquaredError(), 
                  'dense_24': tf.keras.metrics.RootMeanSquaredError(),
                  'dense_25': tf.keras.metrics.RootMeanSquaredError(),
                  'dense_26': tf.keras.metrics.RootMeanSquaredError(),
                  'dense_27': tf.keras.metrics.RootMeanSquaredError(),
                  'dense_28': tf.keras.metrics.RootMeanSquaredError(),
                  'dense_29': tf.keras.metrics.RootMeanSquaredError()})
model.fit(train_dataset, epochs=3, batch_size=batch_sz)